In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Configuration
VOLUME_BASE = "/Volumes/imdb_final_project/raw/raw_store"
ENABLE_SCHEMA_EVOLUTION = False

# Helper function to reduce code duplication
def create_bronze_table(subfolder, schema_hints):
    """Helper to create bronze ingestion logic"""
    reader = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.schemaLocation", f"{VOLUME_BASE}/{subfolder}/_schema_checkpoint")
            .option("cloudFiles.inferColumnTypes", "true")
            .option("cloudFiles.schemaHints", schema_hints)
            .option("delimiter", "\t")
            .option("header", "true")
            .option("multiLine", "true")
            .option("escape", "\"")
            .option("nullValue", "\\N")
    )
    
    if ENABLE_SCHEMA_EVOLUTION:
        reader = reader.option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    
    df = reader.load(f"{VOLUME_BASE}/{subfolder}")
    
    # # Rename columns
    # for col_name in df.columns:
    #     clean_name = col_name.replace(" ", "_").replace("#", "Number")
    #     df = df.withColumnRenamed(col_name, clean_name)
    

    import re
    for col_name in df.columns:
        # Replace invalid chars with underscore, then clean up multiple underscores
        clean_name = re.sub(r'[ ,;{}()\n\t=]+', '_', col_name)
        clean_name = clean_name.replace("#", "Number")
        # Remove leading/trailing underscores
        clean_name = clean_name.strip('_')
        df = df.withColumnRenamed(col_name, clean_name)

    # Add audit columns
    return (
        df
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("ingestion_date", current_date())
    )

# ============================================
# BRONZE TABLES
# ============================================

@dlt.table(
    name="bronze_name_basics_raw",
    comment="Raw IMDB name.basics data"
)
def bronze_name_basics():
    return create_bronze_table("name_basics", "nconst STRING, birthYear STRING, deathYear STRING")

@dlt.table(
    name="bronze_title_akas_raw",
    comment="Raw IMDB title.akas data"
)
def bronze_title_akas():
    return create_bronze_table("title_akas", "titleId STRING, ordering STRING")

@dlt.table(
    name = "bronze_title_crew_raw",
    comment = "Raw IMDB title.crew data"
)
def bronze_title_crew():
    return create_bronze_table("title_crew", "tconst STRING, directors STRING, writers STRING")

@dlt.table(
    name="bronze_title_ratings_raw",
    comment="Raw IMDB title.ratings data"
)
def bronze_title_ratings():
    return create_bronze_table("title_ratings", "tconst STRING, averageRating STRING, numVotes STRING")

@dlt.table(
    name = "bronze_title_region_raw",
    comment = "Raw IMDB title.region data"
)
def bronze_title_region():
    return create_bronze_table("title_region", "tconst STRING, region STRING"
)
    
@dlt.table(
    name="bronze_title_episode_raw",
    comment="Raw IMDB title.episode data"
)
def bronze_title_episode():
    return create_bronze_table("title_episode", "tconst STRING, parentTconst STRING, seasonNumber STRING, episodeNumber STRING")

@dlt.table(
    name="bronze_title_principals_raw",
    comment="Raw IMDB title.principals data"
)
def bronze_title_principals():
    return create_bronze_table("title_principals", "tconst STRING, ordering STRING")

@dlt.table(
    name="bronze_title_basics_raw",
    comment="Raw IMDB title.basics data"
)
def bronze_title_basics():
    return create_bronze_table("title_basics", "tconst STRING, titleType STRING, primaryTitle STRING, originalTitle STRING")


@dlt.table(
    name = "bronze_title_language_codes_raw",
    comment = "Raw IMDB title.language_codes data"
)

def bronze_title_language_codes():
    return create_bronze_table("language_codes", "Language_name STRING, language_code STRING")


In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

@dlt.table(
    name="silver.silver_name_basics",
    comment="Cleaned and exploded IMDB name basics data - preserves all records",
    table_properties={
        "quality": "silver",    
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect_all_or_drop({
    "valid_nconst": "NCONST IS NOT NULL AND NCONST RLIKE '^nm[0-9]{7,}$'",
    # "valid_primary_name": "PRIMARY_NAME IS NOT NULL AND LENGTH(TRIM(PRIMARY_NAME)) > 0"
})
def silver_name_basics():
    df = dlt.read_stream("bronze_name_basics_raw")
    
    # Replace \N with NULLs
    df = df.select(
        *[when(col(c) == "\\N", None).otherwise(col(c)).alias(c) for c in df.columns]
    )
    
    # Rename columns to uppercase
    df = (
        df
        .withColumnRenamed("nconst", "NCONST")
        .withColumnRenamed("primaryName", "PRIMARY_NAME")
        .withColumnRenamed("birthYear", "BIRTH_YEAR")
        .withColumnRenamed("deathYear", "DEATH_YEAR")
        .withColumnRenamed("primaryProfession", "PRIMARY_PROFESSION")
        .withColumnRenamed("knownForTitles", "KNOWN_FOR_TITLES")
    )
    
    # Handle NULL values
    df = (
        df
        .withColumn("BIRTH_YEAR", 
                   when(col("BIRTH_YEAR").isNull(), "0000")
                   .otherwise(col("BIRTH_YEAR")))
        .withColumn("DEATH_YEAR", 
                   when(col("DEATH_YEAR").isNull(), "9999")
                   .otherwise(col("DEATH_YEAR")))
        .withColumn("PRIMARY_PROFESSION", 
                   when(col("PRIMARY_PROFESSION").isNull(), "Unknown")
                   .otherwise(col("PRIMARY_PROFESSION")))
        .withColumn("KNOWN_FOR_TITLES", 
                   when(col("KNOWN_FOR_TITLES").isNull(), "Unknown")
                   .otherwise(col("KNOWN_FOR_TITLES")))
    )
    
    # Cast to integer
    df = (
        df
        .withColumn("BIRTH_YEAR", col("BIRTH_YEAR").cast("int"))
        .withColumn("DEATH_YEAR", col("DEATH_YEAR").cast("int"))
    )
    
    # Add is_alive flag
    df = df.withColumn("IS_ALIVE", when(col("DEATH_YEAR") == 9999, True).otherwise(False))
    
    # Split arrays
    df = (
        df
        .withColumn("PRIMARY_PROFESSION_ARRAY", split(col("PRIMARY_PROFESSION"), ","))
        .withColumn("KNOWN_FOR_TITLES_ARRAY", split(col("KNOWN_FOR_TITLES"), ","))
    )
    
    # Explode primaryProfession
    df_profession_exploded = (
        df
        .withColumn("PROFESSION", explode(col("PRIMARY_PROFESSION_ARRAY")))
        .drop("PRIMARY_PROFESSION_ARRAY")
    )
    
    # Explode knownForTitles
    df_fully_exploded = (
        df_profession_exploded
        .withColumn("KNOWN_FOR_TITLE", explode(col("KNOWN_FOR_TITLES_ARRAY")))
        .drop("KNOWN_FOR_TITLES_ARRAY")
    )
    
    # Trim whitespace
    df_fully_exploded = (
        df_fully_exploded
        .withColumn("PROFESSION", trim(col("PROFESSION")))
        .withColumn("KNOWN_FOR_TITLE", trim(col("KNOWN_FOR_TITLE")))
    )
    
    # Add silver processing timestamp
    df_fully_exploded = df_fully_exploded.withColumn(
        "silver_processing_timestamp", 
        current_timestamp()
    )
    
    # Select final columns
    return df_fully_exploded.select(
        "NCONST",
        "PRIMARY_NAME",
        "BIRTH_YEAR",
        "DEATH_YEAR",
        "IS_ALIVE",
        "PROFESSION",
        "KNOWN_FOR_TITLE",
        "PRIMARY_PROFESSION",
        "KNOWN_FOR_TITLES",
        "ingestion_timestamp",
        "silver_processing_timestamp",
        "source_file",
        "ingestion_date"
    )

In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

@dlt.table(
    name="silver.silver_title_akas",
    comment="Cleaned IMDB title.akas data with NULL handling - no explosion needed",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect_all_or_drop({
    "valid_titleId": "TITLE_ID IS NOT NULL AND TITLE_ID RLIKE '^tt[0-9]{7,}$'",
    "valid_ordering": "ORDERING >= -1"
})
def silver_title_akas():
    """
    Cleans the bronze title_akas data.
    
    Transformations:
    - Replace \\N with NULLs
    - Replace NULL ordering with -1
    - Replace NULL text fields with 'Unknown'
    - Replace NULL isOriginalTitle with -1
    - Create boolean flag for isOriginalTitle
    - Trim whitespace
    """
    
    df = dlt.read_stream("bronze_title_akas_raw")
    
    # Replace \N with NULLs
    df = df.select(
        *[when(col(c) == "\\N", None).otherwise(col(c)).alias(c) for c in df.columns]
    )
    
    # Rename columns to uppercase
    df = (
        df
        .withColumnRenamed("titleId", "TITLE_ID")
        .withColumnRenamed("ordering", "ORDERING")
        .withColumnRenamed("title", "TITLE")
        .withColumnRenamed("region", "REGION")
        .withColumnRenamed("language", "LANGUAGE")
        .withColumnRenamed("types", "TYPES")
        .withColumnRenamed("attributes", "ATTRIBUTES")
        .withColumnRenamed("isOriginalTitle", "IS_ORIGINAL_TITLE")
    )
    
    # Handle NULL values
    df = (
        df
        .withColumn("ORDERING", 
                   when(col("ORDERING").isNull(), "-1")
                   .otherwise(col("ORDERING")))
        .withColumn("TITLE", 
                   when(col("TITLE").isNull(), "Unknown")
                   .otherwise(col("TITLE")))
        .withColumn("REGION", 
                   when(col("REGION").isNull(), "Unknown")
                   .otherwise(col("REGION")))
        .withColumn("LANGUAGE", 
                   when(col("LANGUAGE").isNull(), "Unknown")
                   .otherwise(col("LANGUAGE")))
        .withColumn("TYPES", 
                   when(col("TYPES").isNull(), "Unknown")
                   .otherwise(col("TYPES")))
        .withColumn("ATTRIBUTES", 
                   when(col("ATTRIBUTES").isNull(), "Unknown")
                   .otherwise(col("ATTRIBUTES")))
        .withColumn("IS_ORIGINAL_TITLE", 
                   when(col("IS_ORIGINAL_TITLE").isNull(), "-1")
                   .otherwise(col("IS_ORIGINAL_TITLE")))
    )
    
    # Cast to integer
    df = (
        df
        .withColumn("ORDERING", col("ORDERING").cast("int"))
        .withColumn("IS_ORIGINAL_TITLE", col("IS_ORIGINAL_TITLE").cast("int"))
    )
    
    # Create boolean flag for original title
    df = df.withColumn("IS_ORIGINAL_TITLE_FLAG",
                      when(col("IS_ORIGINAL_TITLE") == 1, True)
                      .when(col("IS_ORIGINAL_TITLE") == 0, False)
                      .otherwise(None))
    
    # Trim whitespace
    df = (
        df
        .withColumn("TITLE", trim(col("TITLE")))
        .withColumn("REGION", trim(col("REGION")))
        .withColumn("LANGUAGE", trim(col("LANGUAGE")))
        .withColumn("TYPES", trim(col("TYPES")))
        .withColumn("ATTRIBUTES", trim(col("ATTRIBUTES")))
    )
    
    # Add silver processing timestamp
    df = df.withColumn(
        "silver_processing_timestamp", 
        current_timestamp()
    )
    
    # Select final columns
    return df.select(
        "TITLE_ID",
        "ORDERING",
        "TITLE",
        "REGION",
        "LANGUAGE",
        "TYPES",
        "ATTRIBUTES",
        "IS_ORIGINAL_TITLE",
        "IS_ORIGINAL_TITLE_FLAG",
        "ingestion_timestamp",
        "silver_processing_timestamp",
        "source_file",
        "ingestion_date"
    )

In [0]:
@dlt.table(
    name="silver.silver_title_language_codes",
    comment="Cleaned IMDB language codes lookup table - preserves all records",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
def silver_title_language_codes():
    df = dlt.read_stream("bronze_title_language_codes_raw")
    
    # # Replace \N with NULLs
    # df = df.select(
    #     *[when(col(c) == "\\N", None).otherwise(col(c)).alias(c) for c in df.columns]
    # )
    
    # Rename columns to uppercase
    df = (
        df
        .withColumnRenamed("Language_Name", "LANGUAGE_NAME")
        .withColumnRenamed("Language_CODE", "LANGUAGE_CODE")
    )
    
    # # Handle NULL values
    # df = (
    #     df
    #     .withColumn("LANGUAGE_NAME", 
    #                when(col("LANGUAGE_NAME").isNull(), "Unknown")
    #                .otherwise(col("LANGUAGE_NAME")))
    #     .withColumn("LANGUAGE_CODE", 
    #                when(col("LANGUAGE_CODE").isNull(), "unknown")
    #                .otherwise(col("LANGUAGE_CODE")))
    # )
    
    # # Trim whitespace
    # df = (
    #     df
    #     .withColumn("LANGUAGE_NAME", trim(col("LANGUAGE_NAME")))
    #     .withColumn("LANGUAGE_CODE", trim(col("LANGUAGE_CODE")))
    # )
    
    # Add silver processing timestamp
    df = df.withColumn(
        "silver_processing_timestamp", 
        current_timestamp()
    )
    
    # Select final columns
    return df.select(
        "LANGUAGE_NAME",
        "LANGUAGE_CODE",
        "ingestion_timestamp",
        "silver_processing_timestamp",
        "source_file",
        "ingestion_date"
    )